In [ ]:
#第一步：训练模型
import os
import math
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
import json
from utils import utils_image as util
from utils import utils_option as option
from data.select_dataset import define_Dataset
from models.select_model import define_Model
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")


# ========================================
# 硬编码配置
# ========================================
def get_config():
    """训练配置"""
    config = {
        # 基本设置
        "task": "Guided SR",
        "model": "plain",
        "gpu_ids": [0],
        "dist": False,
        "scale": 8,
        "n_channels_lr": 1,
        "n_channels_guide": 1,
        "wandb": False,
        
        # 路径设置
        "path": {
            "root": "F:/ChaoFen/Result/IR_VIS_SR_x8",
            "images": "F:/ChaoFen/Result/IR_VIS_SR_x8/images",#保存测试结果
            "models": "F:/ChaoFen/Result/IR_VIS_SR_x8/models"#保存训练好的模型
        },
        
        # 数据集
        "datasets": {
            "train": {
                "name": "train_dataset",
                "dataset_type": "guided_sr",
                "phase": "train",
                "dataroot_lr": "F:/ChaoFen/Datasets/Train/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Train/Visible/",
                "dataroot_gt": "F:/ChaoFen/Datasets/Train/infrared/GT/",
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": True,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": 90000,  
                "manual_seed": 60     
            },
            "validation": {
                "name": "validation_dataset",
                "dataset_type": "guided_sr",
                "phase": "validation",
                "dataroot_lr": "F:/ChaoFen/Datasets/Val/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Val/visible/",
                "dataroot_gt": "F:/ChaoFen/Datasets/Val/infrared/GT/",
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": False,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": 5000,  
                "manual_seed": 60     
            },
            "test": {
                "name": "test_dataset",
                "dataset_type": "guided_sr",
                "phase": "test",
                #"dataroot_lr": "F:/ChaoFen/Datasets/Test/infrared/LR_x8/", 
                #"dataroot_guide": "F:/ChaoFen/Datasets/Test/visible/",
                #"dataroot_gt": "F:/ChaoFen/Datasets/Test/infrared/GT/",
                "dataroot_lr": "F:/ChaoFen/Datasets/Val/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Val/visible/",
                "dataroot_gt": "F:/ChaoFen/Datasets/Val/infrared/GT/",
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": False,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": 20000,    
                "manual_seed": 60
            }
        },
        
        # 网络结构
        "netG": {
            "net_type": "swinfusionSR",#模型使用swinfusionSR
            "upscale": 8,
            "in_chans": 1,
            "img_size": 128,#要改的话也得改成8的倍数
            "window_size": 8,
            "img_range": 1.0, #不要改
            "embed_dim": 6, #卷积核的个数 可以改小一下
            "Ex_depths": [2],#向量的长度代表forward_features_EX_A与forward_features_EX_B用几个basic layer，一个basic_layer即为一个RSTB，元素的值代表每个basic_layer用几层SwinTransformer 这个设置可以不用改
            "Ex_num_heads": [1],#forward_features_Re中每个RSTB的SwinTransformerBlock中的注意力的heads个数
            "Fusion_depths": [2],#向量的长度代表用几个CRSTB block 这个设置可以不用改
            "Fusion_num_heads": [1],
            "Re_depths": [2],#向量的长度代表forward_features_Re中使用几个RSTB
            "Re_num_heads": [1],
            "mlp_ratio": 2,
            "upsampler": "",
            "resi_connection": "1conv",
            "init_type": "default"
        },
        
        # 训练设置
        "train": {
            "batch_size": 50,
            "G_lossfn_type": "l1",
            "weights": {
                "l1": 1.0,
                "mse": 0.0,
                "ssim": 0.0,
                "psnr": 0.0,
                "contrast": 0.0,#如果非0就代表在模型中加入self.contrastive_head1 这里设置0不要去改
                "lpips": 0.0,
                "adversarial": 0.0
            },
            "manual_seed": 60,
            "G_lossfn_weight": 1.0,
            "E_decay": 0.999,
            "G_optimizer_type": "adam",
            "G_optimizer_lr": 1e-3,
            "G_optimizer_wd": 0,
            "G_optimizer_clipgrad": None,
            "G_optimizer_reuse": True,
            "G_scheduler_type": "MultiStepLR",
            "G_scheduler_milestones": [150000, 250000],
            "G_scheduler_gamma": 0.5,
            "G_regularizer_orthstep": None,
            "G_regularizer_clipstep": None,
            "G_param_strict": True,
            "E_param_strict": True,
            "checkpoint_test": 36000,#多少个batch计算一次验证样本的准确性
            "checkpoint_print": 36000,#多少个batch打印一次训练信息
            "limit_test": 1000000,
            "limit_validation": 1000000,
            "epochs": 200
        },
        
        "rank": 0,
        "world_size": 1,
        "num_gpu": 1
    }
    
    return config

def plot_ssim_curve(val_steps, val_ssim_history, save_path):
    plt.figure(figsize=(12, 6))
    plt.plot(val_steps, val_ssim_history, 'b-o', linewidth=2, markersize=6, label='Validation SSIM')
    best_ssim = max(val_ssim_history)
    best_step = val_steps[val_ssim_history.index(best_ssim)]
    plt.plot(best_step, best_ssim, 'r*', markersize=15, label=f'Best: {best_ssim:.4f} @ Step {best_step}')
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.xlabel('Training Step', fontsize=14, fontweight='bold')
    plt.ylabel('SSIM', fontsize=14, fontweight='bold')
    plt.title('Validation SSIM Curve', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.ylim([max(0, min(val_ssim_history)-0.02), 1.0])
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ SSIM曲线已保存: {save_path}")
    plt.close()
        
def main():
    """主训练函数"""
    
    print("="*60)
    print("Infrared Image Super-Resolution Training")
    print("="*60)
    # ----------------------------------------
    # GPU 检测
    # ----------------------------------------
    # 获取配置
    opt = get_config()
    
    # 创建目录
    for path in opt['path'].values():
        os.makedirs(path, exist_ok=True)
    
    # 转换配置
    opt = option.dict_to_nonedict(opt)
    
    # 设置随机种子
    seed = opt['train']['manual_seed']
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"Random seed: {seed}")
    
    # ----------------------------------------
    # 创建数据加载器
    # ----------------------------------------
    print("\n加载数据集...")
    
    # 训练集
    train_set = define_Dataset(opt['datasets']['train'])
    train_loader = DataLoader(
        train_set,
        batch_size=opt["train"]['batch_size'],
        shuffle=opt['datasets']['train']['dataloader_shuffle'],
        num_workers=4,
        drop_last=True,
        pin_memory=True,
        persistent_workers=True
    )
    print(f"训练集: {len(train_set)} 张图像")
    
    # 验证集
    val_set = define_Dataset(opt['datasets']['validation'])
    val_loader = DataLoader(val_set, 
                            batch_size=1, 
                            shuffle=False, 
                            num_workers=2, 
                            pin_memory=True)
    print(f"验证集: {len(val_set)} 张图像")
    
    # 测试集
    test_set = define_Dataset(opt['datasets']['test'])
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
    print(f"测试集: {len(test_set)} 张图像")
    
    # ----------------------------------------
    # 初始化模型
    # ----------------------------------------
    print("\n初始化模型...")
    
    # Guide Dropout设置
    guide_dropout_start_iter = 500000
    final_proba_without_rgb = 0.2
    if hasattr(train_loader.dataset, 'proba_without_rgb'):
        train_loader.dataset.proba_without_rgb = 0.0
    
    model = define_Model(opt)
    model.init_train()
    print("模型初始化完成")
    
    # ----------------------------------------
    # 主训练循环
    # ----------------------------------------
    print("\n开始训练...")
    print("="*60)
    
    # 训练历史记录
    val_ssim_history = []
    val_psnr_history = []
    val_steps = []

    current_step = 0
    best_step = 0 
    max_psnr = 0.0

    for epoch in range(opt["train"]["epochs"]):
        # 添加进度条
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{opt["train"]["epochs"]}', 
                unit='batch', ncols=100)
    
        # 训练
        for i, train_data in enumerate(pbar):
            current_step += 1
        
            # 启用Guide Dropout
            if current_step == guide_dropout_start_iter:
                if hasattr(train_loader.dataset, 'proba_without_rgb'):
                    train_loader.dataset.proba_without_rgb = final_proba_without_rgb
                    print(f"\n[步数 {current_step}] 启用Guide Dropout (概率: {final_proba_without_rgb})")
        
            # 训练
            model.feed_data(train_data, need_GT=True, phase="train")
            model.optimize_parameters(current_step, phase="train")

            # 更新进度条显示的信息
            logs = model.current_log()
            lr = model.G_optimizer.param_groups[0]['lr']
        
            # 简化显示信息
            loss_info = {k: f'{v:.4f}' for k, v in logs.items()}
            pbar.set_postfix(loss_info, lr=f'{lr:.2e}')
        
            # 打印训练信息(减少打印频率,避免与进度条冲突)
            if current_step % opt['train']['checkpoint_print'] == 0:
                loss_str = ', '.join([f'{k}: {v:.4f}' for k, v in logs.items()])
                tqdm.write(f"\n[Epoch {epoch+1:4d}] [Step {current_step:8d}] [LR {lr:.2e}] {loss_str}")
        
            # 验证
            if current_step % opt['train']['checkpoint_test'] == 0:
                pbar.close()  # 关闭当前进度条
            
                print(f"\n{'='*60}")
                print(f"验证 (步数: {current_step})")
                print(f"{'='*60}")
            
                # 验证循环 - 也可以加进度条
                total_psnr = 0.0
                total_ssim = 0.0
                val_count = 0
            
                val_pbar = tqdm(val_loader, desc='验证中', unit='img', ncols=100)
                for it, val_data in enumerate(val_pbar):
                    if it >= opt['train']['limit_validation']:
                        break
                
                    model.feed_data(val_data, need_GT=True, phase='validation')
                    model.test()
                
                    visuals = model.current_visuals(need_H=True)
                    E_img = util.tensor2uint(visuals['Output'])
                    H_img = util.tensor2uint(visuals['GT'])
                
                    psnr = util.calculate_psnr(E_img, H_img)
                    ssim = util.calculate_ssim(E_img, H_img)
                    
                    total_psnr += psnr
                    total_ssim += ssim
                    val_count += 1
                
                    # 更新验证进度条
                    val_pbar.set_postfix({'PSNR': f'{psnr:.2f}', 'SSIM': f'{ssim:.4f}'})
            
                val_pbar.close()
            
                avg_psnr = total_psnr / val_count
                avg_ssim = total_ssim / val_count

                # 记录验证历史
                val_steps.append(current_step)
                val_psnr_history.append(avg_psnr)
                val_ssim_history.append(avg_ssim)

                print(f"平均PSNR: {avg_psnr:.2f} dB")
                print(f"平均SSIM: {avg_ssim:.4f}")
            
                # 保存最佳模型
                if avg_psnr > max_psnr:
                    max_psnr = avg_psnr
                    best_step = current_step
                    print(f"*** 新的最佳PSNR: {max_psnr:.2f} dB ***")
                    model.save(current_step)
                    print(f"模型已保存")

                # 重新创建进度条
                pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{opt["train"]["epochs"]}', 
                       unit='batch', ncols=100, initial=i+1, total=len(train_loader))
        
        pbar.close()
        
    # 绘制SSIM曲线
    print("\n" + "="*60)
    print("绘制SSIM曲线...")
    print("="*60)
    if len(val_ssim_history) > 0:
        ssim_curve_path = os.path.join(opt['path']['root'], 'validation_ssim_curve.png')
        plot_ssim_curve(val_steps, val_ssim_history, ssim_curve_path)
    
        # 保存历史数据
        history_path = os.path.join(opt['path']['root'], 'training_history.json')
        with open(history_path, 'w') as f:
            json.dump({'val_steps': val_steps, 'val_ssim': val_ssim_history}, f, indent=4)
        print(f"✓ 训练历史已保存: {history_path}")
        
    # ----------------------------------------
    # 训练完成后,加载最佳模型并测试
    # ----------------------------------------
    print("\n" + "="*60)
    print("训练完成!开始在测试集上生成结果...")
    print("="*60)
    
    # 加载最佳模型
    best_model_path = os.path.join(opt['path']['models'], f'{best_step}_G.pth')
    if os.path.exists(best_model_path):
        print(f"加载最佳模型: {best_model_path}")
        try:
            model.load_network(best_model_path, model.netG, strict=True, param_key='params')
            print("✓ 最佳模型加载成功!")
        except Exception as e:
            print(f"⚠ 加载失败,尝试不使用 param_key: {e}")
    else:
        print(f"⚠ 最佳模型文件未找到: {best_model_path}")
    
    model.netG.eval()
    
    # 测试循环 - 也加进度条
    test_pbar = tqdm(test_loader, desc='测试中', unit='img', ncols=100)
    for it, test_data in enumerate(test_pbar):
        image_name = os.path.basename(test_data['Guide_path'][0])
        model.feed_data(test_data, need_GT=False, phase='test')
        model.test()
        
        visuals = model.current_visuals(need_H=False)
        E_img = util.tensor2uint(visuals['Output'])
        
        save_path = os.path.join(opt['path']['images'], image_name)
        util.imsave(E_img, save_path)
        
        test_pbar.set_postfix({'当前': image_name})
    
    test_pbar.close()
    
    print(f"\n✓ 测试结果已保存到: {opt['path']['images']}")
    print(f"✓ 最佳PSNR: {max_psnr:.2f} dB (步数: {best_step})")
    print("="*60)


if __name__ == '__main__':
    main()

Infrared Image Super-Resolution Training
Random seed: 60

加载数据集...
Dataset: for guided super-resolution.
📊 [train] 原始样本数: 98,700
🎲 [train] 随机采样 90,000 个样本...
✅ [train] 采样后样本数: 90,000
Dataset [Dataset - train_dataset] is created.
训练集: 90000 张图像
Dataset: for guided super-resolution.
📊 [validation] 原始样本数: 33,149
🎲 [validation] 随机采样 5,000 个样本...
✅ [validation] 采样后样本数: 5,000
Dataset [Dataset - validation_dataset] is created.
验证集: 5000 张图像
Dataset: for guided super-resolution.
📊 [test] 原始样本数: 33,149
🎲 [test] 随机采样 20,000 个样本...
✅ [test] 采样后样本数: 20,000
Dataset [Dataset - test_dataset] is created.
测试集: 20000 张图像

初始化模型...
Training model [ModelPlain] is created.
Copying model for E ...
模型初始化完成

开始训练...


Epoch 20/200: 100%|█████████████▉| 1799/1800 [40:13<00:01,  1.34s/batch, G_loss=0.0153, lr=1.00e-03]



[Epoch   20] [Step    36000] [LR 1.00e-03] G_loss: 0.0153

验证 (步数: 36000)


验证中: 100%|█████████████████████████| 5000/5000 [05:34<00:00, 14.97img/s, PSNR=38.37, SSIM=0.9855]


平均PSNR: 32.63 dB
平均SSIM: 0.8792
*** 新的最佳PSNR: 32.63 dB ***
模型已保存


Epoch 40/200: 100%|█████████████▉| 1799/1800 [41:30<00:01,  1.38s/batch, G_loss=0.0132, lr=1.00e-03]


... (2 outputs left out)

平均PSNR: 33.08 dB
平均SSIM: 0.8877
*** 新的最佳PSNR: 33.08 dB ***
模型已保存


Epoch 60/200: 100%|███████████▉| 1799/1800 [1:29:31<00:02,  2.99s/batch, G_loss=0.0149, lr=1.00e-03]



[Epoch   60] [Step   108000] [LR 1.00e-03] G_loss: 0.0149

验证 (步数: 108000)


验证中:   0%|                                                             | 0/5000 [00:00<?, ?img/s]

In [1]:
#第一步：训练模型（不验证）
import os
import math
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
import json
from utils import utils_image as util
from utils import utils_option as option
from data.select_dataset import define_Dataset
from models.select_model import define_Model
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")


# ========================================
# 硬编码配置
# ========================================
def get_config():
    """训练配置"""
    config = {
        # 基本设置
        "task": "Guided SR",
        "model": "plain",
        "gpu_ids": [0],
        "dist": False,
        "scale": 8,
        "n_channels_lr": 1,
        "n_channels_guide": 1,
        "wandb": False,
        
        # 路径设置
        "path": {
            "root": "F:/ChaoFen/Result/IR_VIS_SR_x8",
            "images": "F:/ChaoFen/Result/IR_VIS_SR_x8/images",#保存测试结果
            "models": "F:/ChaoFen/Result/IR_VIS_SR_x8/models"#保存训练好的模型
        },
        
        # 数据集
        "datasets": {
            "train": {
                "name": "train_dataset",
                "dataset_type": "guided_sr",
                "phase": "train",
                "dataroot_lr": "F:/ChaoFen/Datasets/Train/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Train/Visible/",
                "dataroot_gt": "F:/ChaoFen/Datasets/Train/infrared/GT/",
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": True,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": 80000,  
                "manual_seed": 60     
            },
            "validation": {
                "name": "validation_dataset",
                "dataset_type": "guided_sr",
                "phase": "validation",
                "dataroot_lr": "F:/ChaoFen/Datasets/Val/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Val/visible/",
                "dataroot_gt": "F:/ChaoFen/Datasets/Val/infrared/GT/",
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": False,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": 50,  
                "manual_seed": 60     
            },
            "test": {
                "name": "test_dataset",
                "dataset_type": "guided_sr",
                "phase": "test",
                #"dataroot_lr": "F:/ChaoFen/Datasets/Test/infrared/LR_x8/", 
                #"dataroot_guide": "F:/ChaoFen/Datasets/Test/visible/",
                #"dataroot_gt": "F:/ChaoFen/Datasets/Test/infrared/GT/",
                "dataroot_lr": "F:/ChaoFen/Datasets/Val/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Val/visible/",
                "dataroot_gt": "F:/ChaoFen/Datasets/Val/infrared/GT/",
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": False,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": 20000,    
                "manual_seed": 60
            }
        },
        
        # 网络结构
        "netG": {
            "net_type": "swinfusionSR",#模型使用swinfusionSR
            "upscale": 8,
            "in_chans": 1,
            "img_size": 128,#要改的话也得改成8的倍数
            "window_size": 8,
            "img_range": 1.0, #不要改
            "embed_dim": 6, #卷积核的个数 可以改小一下
            "Ex_depths": [2],#向量的长度代表forward_features_EX_A与forward_features_EX_B用几个basic layer，一个basic_layer即为一个RSTB，元素的值代表每个basic_layer用几层SwinTransformer 这个设置可以不用改
            "Ex_num_heads": [1],#forward_features_Re中每个RSTB的SwinTransformerBlock中的注意力的heads个数
            "Fusion_depths": [2],#向量的长度代表用几个CRSTB block 这个设置可以不用改
            "Fusion_num_heads": [1],
            "Re_depths": [2],#向量的长度代表forward_features_Re中使用几个RSTB
            "Re_num_heads": [1],
            "mlp_ratio": 2,
            "upsampler": "",
            "resi_connection": "1conv",
            "init_type": "default"
        },
        
        # 训练设置
        "train": {
            "batch_size": 50,
            "G_lossfn_type": "l1",
            "weights": {
                "l1": 1.0,
                "mse": 0.0,
                "ssim": 0.0,
                "psnr": 0.0,
                "contrast": 0.0,#如果非0就代表在模型中加入self.contrastive_head1 这里设置0不要去改
                "lpips": 0.0,
                "adversarial": 0.0
            },
            "manual_seed": 60,
            "G_lossfn_weight": 1.0,
            "E_decay": 0.999,
            "G_optimizer_type": "adam",
            "G_optimizer_lr": 1e-3,
            "G_optimizer_wd": 0,
            "G_optimizer_clipgrad": None,
            "G_optimizer_reuse": True,
            "G_scheduler_type": "MultiStepLR",
            "G_scheduler_milestones": [150000, 250000],
            "G_scheduler_gamma": 0.5,
            "G_regularizer_orthstep": None,
            "G_regularizer_clipstep": None,
            "G_param_strict": True,
            "E_param_strict": True,
            "checkpoint_test": 16000,#多少个batch计算一次验证样本的准确性
            "checkpoint_print": 16000,#多少个batch打印一次训练信息
            "limit_test": 1000000,
            "limit_validation": 1000000,
            "epochs": 200
        },
        
        "rank": 0,
        "world_size": 1,
        "num_gpu": 1
    }
    
    return config

def plot_ssim_curve(val_steps, val_ssim_history, save_path):
    plt.figure(figsize=(12, 6))
    plt.plot(val_steps, val_ssim_history, 'b-o', linewidth=2, markersize=6, label='Validation SSIM')
    best_ssim = max(val_ssim_history)
    best_step = val_steps[val_ssim_history.index(best_ssim)]
    plt.plot(best_step, best_ssim, 'r*', markersize=15, label=f'Best: {best_ssim:.4f} @ Step {best_step}')
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.xlabel('Training Step', fontsize=14, fontweight='bold')
    plt.ylabel('SSIM', fontsize=14, fontweight='bold')
    plt.title('Validation SSIM Curve', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.ylim([max(0, min(val_ssim_history)-0.02), 1.0])
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ SSIM曲线已保存: {save_path}")
    plt.close()
        
def main():
    """主训练函数"""
    
    print("="*60)
    print("Infrared Image Super-Resolution Training")
    print("="*60)
    # ----------------------------------------
    # GPU 检测
    # ----------------------------------------
    # 获取配置
    opt = get_config()
    
    # 创建目录
    for path in opt['path'].values():
        os.makedirs(path, exist_ok=True)
    
    # 转换配置
    opt = option.dict_to_nonedict(opt)
    
    # 设置随机种子
    seed = opt['train']['manual_seed']
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"Random seed: {seed}")
    
    # ----------------------------------------
    # 创建数据加载器
    # ----------------------------------------
    print("\n加载数据集...")
    
    # 训练集
    train_set = define_Dataset(opt['datasets']['train'])
    train_loader = DataLoader(
        train_set,
        batch_size=opt["train"]['batch_size'],
        shuffle=opt['datasets']['train']['dataloader_shuffle'],
        num_workers=4,
        drop_last=True,
        pin_memory=True,
        persistent_workers=True
    )
    print(f"训练集: {len(train_set)} 张图像")
    
    # 验证集
    val_set = define_Dataset(opt['datasets']['validation'])
    val_loader = DataLoader(val_set, 
                            batch_size=1, 
                            shuffle=False, 
                            num_workers=2, 
                            pin_memory=True)
    print(f"验证集: {len(val_set)} 张图像")
    
    # 测试集
    test_set = define_Dataset(opt['datasets']['test'])
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
    print(f"测试集: {len(test_set)} 张图像")
    
    # ----------------------------------------
    # 初始化模型
    # ----------------------------------------
    print("\n初始化模型...")
    
    # Guide Dropout设置
    guide_dropout_start_iter = 500000
    final_proba_without_rgb = 0.2
    if hasattr(train_loader.dataset, 'proba_without_rgb'):
        train_loader.dataset.proba_without_rgb = 0.0
    
    model = define_Model(opt)
    model.init_train()
    print("模型初始化完成")
    
    # ----------------------------------------
    # 主训练循环
    # ----------------------------------------
    print("\n开始训练...")
    print("="*60)
    
    # 训练历史记录
    val_ssim_history = []
    val_psnr_history = []
    val_steps = []

    current_step = 0
    best_step = 0 
    max_psnr = 0.0

    for epoch in range(opt["train"]["epochs"]):
        # 添加进度条
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{opt["train"]["epochs"]}', 
                unit='batch', ncols=100)
    
        # 训练
        for i, train_data in enumerate(pbar):
            current_step += 1
        
            # 启用Guide Dropout
            if current_step == guide_dropout_start_iter:
                if hasattr(train_loader.dataset, 'proba_without_rgb'):
                    train_loader.dataset.proba_without_rgb = final_proba_without_rgb
                    print(f"\n[步数 {current_step}] 启用Guide Dropout (概率: {final_proba_without_rgb})")
        
            # 训练
            model.feed_data(train_data, need_GT=True, phase="train")
            model.optimize_parameters(current_step, phase="train")

            # 更新进度条显示的信息
            logs = model.current_log()
            lr = model.G_optimizer.param_groups[0]['lr']
        
            # 简化显示信息
            loss_info = {k: f'{v:.4f}' for k, v in logs.items()}
            pbar.set_postfix(loss_info, lr=f'{lr:.2e}')
        
            # 打印训练信息(减少打印频率,避免与进度条冲突)
            if current_step % opt['train']['checkpoint_print'] == 0:
                loss_str = ', '.join([f'{k}: {v:.4f}' for k, v in logs.items()])
                tqdm.write(f"\n[Epoch {epoch+1:4d}] [Step {current_step:8d}] [LR {lr:.2e}] {loss_str}")
        
            # 验证
            if current_step % opt['train']['checkpoint_test'] == 0:
                pbar.close()  # 关闭当前进度条
                model.save(current_step)
                print(f"模型已保存")
                #重新创建进度条
                pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{opt["train"]["epochs"]}', 
                       unit='batch', ncols=100, initial=i+1, total=len(train_loader))
        
        pbar.close()
        
    # 绘制SSIM曲线
    print("\n" + "="*60)
    print("绘制SSIM曲线...")
    print("="*60)
    if len(val_ssim_history) > 0:
        ssim_curve_path = os.path.join(opt['path']['root'], 'validation_ssim_curve.png')
        plot_ssim_curve(val_steps, val_ssim_history, ssim_curve_path)
    
        # 保存历史数据
        history_path = os.path.join(opt['path']['root'], 'training_history.json')
        with open(history_path, 'w') as f:
            json.dump({'val_steps': val_steps, 'val_ssim': val_ssim_history}, f, indent=4)
        print(f"✓ 训练历史已保存: {history_path}")
        
    # ----------------------------------------
    # 训练完成后,加载最佳模型并测试
    # ----------------------------------------
    print("\n" + "="*60)
    print("训练完成!开始在测试集上生成结果...")
    print("="*60)
    
    # 加载最佳模型
    best_model_path = os.path.join(opt['path']['models'], f'{best_step}_G.pth')
    if os.path.exists(best_model_path):
        print(f"加载最佳模型: {best_model_path}")
        try:
            model.load_network(best_model_path, model.netG, strict=True, param_key='params')
            print("✓ 最佳模型加载成功!")
        except Exception as e:
            print(f"⚠ 加载失败,尝试不使用 param_key: {e}")
    else:
        print(f"⚠ 最佳模型文件未找到: {best_model_path}")
    
    model.netG.eval()
    
    # 测试循环 - 也加进度条
    test_pbar = tqdm(test_loader, desc='测试中', unit='img', ncols=100)
    for it, test_data in enumerate(test_pbar):
        image_name = os.path.basename(test_data['Guide_path'][0])
        model.feed_data(test_data, need_GT=False, phase='test')
        model.test()
        
        visuals = model.current_visuals(need_H=False)
        E_img = util.tensor2uint(visuals['Output'])
        
        save_path = os.path.join(opt['path']['images'], image_name)
        util.imsave(E_img, save_path)
        
        test_pbar.set_postfix({'当前': image_name})
    
    test_pbar.close()
    
    print(f"\n✓ 测试结果已保存到: {opt['path']['images']}")
    print(f"✓ 最佳PSNR: {max_psnr:.2f} dB (步数: {best_step})")
    print("="*60)


if __name__ == '__main__':
    main()

Infrared Image Super-Resolution Training
Random seed: 60

加载数据集...
Dataset: for guided super-resolution.
📊 [train] 原始样本数: 98,700
🎲 [train] 随机采样 80,000 个样本...
✅ [train] 采样后样本数: 80,000
Dataset [Dataset - train_dataset] is created.
训练集: 80000 张图像
Dataset: for guided super-resolution.
📊 [validation] 原始样本数: 33,149
🎲 [validation] 随机采样 50 个样本...
✅ [validation] 采样后样本数: 50
Dataset [Dataset - validation_dataset] is created.
验证集: 50 张图像
Dataset: for guided super-resolution.
📊 [test] 原始样本数: 33,149
🎲 [test] 随机采样 20,000 个样本...
✅ [test] 采样后样本数: 20,000
Dataset [Dataset - test_dataset] is created.
测试集: 20000 张图像

初始化模型...
Training model [ModelPlain] is created.
Copying model for E ...
模型初始化完成

开始训练...


Epoch 10/200: 100%|███████████▉| 1599/1600 [1:05:12<00:02,  2.45s/batch, G_loss=0.0218, lr=1.00e-03]



[Epoch   10] [Step    16000] [LR 1.00e-03] G_loss: 0.0218
模型已保存


Epoch 20/200: 100%|███████████▉| 1599/1600 [2:02:59<00:04,  4.62s/batch, G_loss=0.0155, lr=1.00e-03]



[Epoch   20] [Step    32000] [LR 1.00e-03] G_loss: 0.0155
模型已保存


Epoch 30/200: 100%|█████████████▉| 1599/1600 [36:58<00:01,  1.39s/batch, G_loss=0.0146, lr=1.00e-03]


... (21 outputs left out)

Epoch 140/200: 100%|████████████▉| 1599/1600 [58:43<00:02,  2.20s/batch, G_loss=0.0182, lr=1.00e-03]



[Epoch  140] [Step   224000] [LR 1.00e-03] G_loss: 0.0182
模型已保存


Epoch 142/200:  88%|███████████▌ | 1416/1600 [52:16<06:47,  2.21s/batch, G_loss=0.0156, lr=1.00e-03]


KeyboardInterrupt: 

In [5]:
#第一步（计算每个pth文件的验证集性能）
"""
评估所有已训练模型在验证集上的性能
"""
import os
import glob
import re
import numpy as np
import torch
from torch.utils.data import DataLoader
import json
from utils import utils_image as util
from utils import utils_option as option
from data.select_dataset import define_Dataset
from models.select_model import define_Model
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")


def get_config():
    """评估配置"""
    config = {
        # 基本设置
        "task": "Guided SR",
        "model": "plain",
        "gpu_ids": [0],
        "dist": False,
        "scale": 8,
        "n_channels_lr": 1,
        "n_channels_guide": 1,
        "wandb": False,
        
        # 路径设置
        "path": {
            "root": "F:/ChaoFen/Result/IR_VIS_SR_x8",
            "models": "F:/ChaoFen/Result/IR_VIS_SR_x8/models"
        },
        
        # 验证数据集
        "datasets": {
            "validation": {
                "name": "validation_dataset",
                "dataset_type": "guided_sr",
                "phase": "validation",
                "dataroot_lr": "F:/ChaoFen/Datasets/Val/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Val/visible/",
                "dataroot_gt": "F:/ChaoFen/Datasets/Val/infrared/GT/",
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": False,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": 10000,  # 验证集样本数
                "manual_seed": 60     
            }
        },
        
       # 网络结构
        "netG": {
            "net_type": "swinfusionSR",#模型使用swinfusionSR
            "upscale": 8,
            "in_chans": 1,
            "img_size": 128,#要改的话也得改成8的倍数
            "window_size": 8,
            "img_range": 1.0, #不要改
            "embed_dim": 6, #卷积核的个数 可以改小一下
            "Ex_depths": [2],#向量的长度代表forward_features_EX_A与forward_features_EX_B用几个basic layer，一个basic_layer即为一个RSTB，元素的值代表每个basic_layer用几层SwinTransformer 这个设置可以不用改
            "Ex_num_heads": [1],#forward_features_Re中每个RSTB的SwinTransformerBlock中的注意力的heads个数
            "Fusion_depths": [2],#向量的长度代表用几个CRSTB block 这个设置可以不用改
            "Fusion_num_heads": [1],
            "Re_depths": [2],#向量的长度代表forward_features_Re中使用几个RSTB
            "Re_num_heads": [1],
            "mlp_ratio": 2,
            "upsampler": "",
            "resi_connection": "1conv",
            "init_type": "default"
        },
        
        # 训练设置
        "train": {
            "batch_size": 1,
            "G_lossfn_type": "l1",
            "weights": {
                "l1": 1.0,
                "mse": 0.0,
                "ssim": 0.0,
                "psnr": 0.0,
                "contrast": 0.0,
                "lpips": 0.0,
                "adversarial": 0.0
            },
            "manual_seed": 60,
            "G_lossfn_weight": 1.0,
            "E_decay": 0.999,
            "G_optimizer_type": "adam",
            "G_optimizer_lr": 1e-3,
            "G_optimizer_wd": 0,
            "G_optimizer_clipgrad": None,
            "G_optimizer_reuse": True,
            "G_scheduler_type": "MultiStepLR",
            "G_scheduler_milestones": [150000, 250000],
            "G_scheduler_gamma": 0.5,
            "G_regularizer_orthstep": None,
            "G_regularizer_clipstep": None,
            "G_param_strict": True,
            "E_param_strict": True,
            "checkpoint_test": 2000,
            "checkpoint_print": 2000,
            "limit_test": 1000000,
            "limit_validation": 1000000,
            "epochs": 200
        },
        
        "rank": 0,
        "world_size": 1,
        "num_gpu": 1
    }
    
    return config


def extract_step_from_filename(filename):
    """从文件名中提取训练步数
    例如: '16000_G.pth' -> 16000
    """
    match = re.search(r'(\d+)_G\.pth', filename)
    if match:
        return int(match.group(1))
    return None


def evaluate_model(model, val_loader, model_path):
    """评估单个模型
    
    Args:
        model: 模型对象
        val_loader: 验证数据加载器
        model_path: 模型文件路径
        
    Returns:
        dict: {'psnr': float, 'ssim': float}
    """
    # 加载模型权重
    try:
        model.load_network(model_path, model.netG, strict=True, param_key='params')
    except:
        # 如果加载失败,尝试不使用param_key
        try:
            checkpoint = torch.load(model_path, map_location=lambda storage, loc: storage)
            model.netG.load_state_dict(checkpoint, strict=True)
        except Exception as e:
            print(f"⚠ 加载模型失败: {e}")
            return None
    
    model.netG.eval()
    
    # 评估
    total_psnr = 0.0
    total_ssim = 0.0
    val_count = 0
    
    with torch.no_grad():
        for val_data in val_loader:
            model.feed_data(val_data, need_GT=True, phase='validation')
            model.test()
            
            visuals = model.current_visuals(need_H=True)
            E_img = util.tensor2uint(visuals['Output'])
            H_img = util.tensor2uint(visuals['GT'])
            
            psnr = util.calculate_psnr(E_img, H_img)
            ssim = util.calculate_ssim(E_img, H_img)
            
            total_psnr += psnr
            total_ssim += ssim
            val_count += 1
    
    avg_psnr = total_psnr / val_count
    avg_ssim = total_ssim / val_count
    
    return {'psnr': avg_psnr, 'ssim': avg_ssim}


def plot_metrics_curves(results, save_dir):
    """绘制PSNR和SSIM曲线"""
    
    steps = [r['step'] for r in results]
    psnrs = [r['psnr'] for r in results]
    ssims = [r['ssim'] for r in results]
    
    # 找到最佳值
    best_psnr_idx = np.argmax(psnrs)
    best_ssim_idx = np.argmax(ssims)
    
    # 创建图表
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # PSNR曲线
    ax1.plot(steps, psnrs, 'b-o', linewidth=2, markersize=6, label='PSNR')
    ax1.plot(steps[best_psnr_idx], psnrs[best_psnr_idx], 'r*', 
             markersize=15, label=f'Best: {psnrs[best_psnr_idx]:.2f} dB @ Step {steps[best_psnr_idx]}')
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.set_xlabel('Training Step', fontsize=14, fontweight='bold')
    ax1.set_ylabel('PSNR (dB)', fontsize=14, fontweight='bold')
    ax1.set_title('Validation PSNR Curve', fontsize=16, fontweight='bold')
    ax1.legend(fontsize=12)
    
    # SSIM曲线
    ax2.plot(steps, ssims, 'g-o', linewidth=2, markersize=6, label='SSIM')
    ax2.plot(steps[best_ssim_idx], ssims[best_ssim_idx], 'r*', 
             markersize=15, label=f'Best: {ssims[best_ssim_idx]:.4f} @ Step {steps[best_ssim_idx]}')
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.set_xlabel('Training Step', fontsize=14, fontweight='bold')
    ax2.set_ylabel('SSIM', fontsize=14, fontweight='bold')
    ax2.set_title('Validation SSIM Curve', fontsize=16, fontweight='bold')
    ax2.legend(fontsize=12)
    ax2.set_ylim([max(0, min(ssims)-0.02), 1.0])
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, 'validation_metrics_curves.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ 指标曲线已保存: {save_path}")
    plt.close()


def main():
    """主评估函数"""
    
    print("="*60)
    print("模型评估: 计算所有checkpoint在验证集上的指标")
    print("="*60)
    
    # 获取配置
    opt = get_config()
    opt = option.dict_to_nonedict(opt)
    
    # ----------------------------------------
    # 加载验证集
    # ----------------------------------------
    print("\n加载验证集...")
    val_set = define_Dataset(opt['datasets']['validation'])
    val_loader = DataLoader(
        val_set, 
        batch_size=1, 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True
    )
    print(f"验证集: {len(val_set)} 张图像")
    
    # ----------------------------------------
    # 初始化模型(仅结构,不加载权重)
    # ----------------------------------------
    print("\n初始化模型结构...")
    model = define_Model(opt)
    print("模型结构初始化完成")
    
    # ----------------------------------------
    # 扫描所有模型文件
    # ----------------------------------------
    models_dir = opt['path']['models']
    model_files = glob.glob(os.path.join(models_dir, '*_G.pth'))
    
    if len(model_files) == 0:
        print(f"⚠ 未找到任何模型文件: {models_dir}")
        return
    
    # 按步数排序
    model_info = []
    for model_file in model_files:
        step = extract_step_from_filename(os.path.basename(model_file))
        if step is not None:
            model_info.append({'path': model_file, 'step': step})
    
    model_info = sorted(model_info, key=lambda x: x['step'])
    
    print(f"\n找到 {len(model_info)} 个模型文件:")
    for info in model_info:
        print(f"  - Step {info['step']}: {os.path.basename(info['path'])}")
    
    # ----------------------------------------
    # 逐个评估模型
    # ----------------------------------------
    print("\n" + "="*60)
    print("开始评估...")
    print("="*60)
    
    results = []
    
    for info in model_info:
        step = info['step']
        model_path = info['path']
        
        print(f"\n[Step {step}] 评估中...")
        
        metrics = evaluate_model(model, val_loader, model_path)
        
        if metrics is not None:
            results.append({
                'step': step,
                'psnr': metrics['psnr'],
                'ssim': metrics['ssim'],
                'model_path': model_path
            })
            print(f"  PSNR: {metrics['psnr']:.2f} dB")
            print(f"  SSIM: {metrics['ssim']:.4f}")
        else:
            print(f"  ⚠ 评估失败")
    
    # ----------------------------------------
    # 保存结果
    # ----------------------------------------
    print("\n" + "="*60)
    print("保存评估结果...")
    print("="*60)
    
    if len(results) == 0:
        print("⚠ 没有成功评估的模型")
        return
    
    # 找到最佳模型
    best_psnr_result = max(results, key=lambda x: x['psnr'])
    best_ssim_result = max(results, key=lambda x: x['ssim'])
    
    # 打印汇总
    print("\n" + "="*60)
    print("评估结果汇总")
    print("="*60)
    print(f"\n{'Step':<10} {'PSNR (dB)':<12} {'SSIM':<10}")
    print("-"*35)
    for r in results:
        marker_psnr = " ★" if r == best_psnr_result else ""
        marker_ssim = " ★" if r == best_ssim_result else ""
        print(f"{r['step']:<10} {r['psnr']:<12.2f}{marker_psnr:<3} {r['ssim']:<10.4f}{marker_ssim}")
    
    print("\n" + "="*60)
    print(f"最佳PSNR: {best_psnr_result['psnr']:.2f} dB @ Step {best_psnr_result['step']}")
    print(f"最佳SSIM: {best_ssim_result['ssim']:.4f} @ Step {best_ssim_result['step']}")
    print("="*60)
    
    # 保存JSON
    result_json_path = os.path.join(opt['path']['root'], 'evaluation_results.json')
    with open(result_json_path, 'w') as f:
        json.dump({
            'results': results,
            'best_psnr': {
                'step': best_psnr_result['step'],
                'psnr': best_psnr_result['psnr'],
                'ssim': best_psnr_result['ssim']
            },
            'best_ssim': {
                'step': best_ssim_result['step'],
                'psnr': best_ssim_result['psnr'],
                'ssim': best_ssim_result['ssim']
            }
        }, f, indent=4)
    print(f"\n✓ 评估结果已保存: {result_json_path}")
    
    # 绘制曲线
    plot_metrics_curves(results, opt['path']['root'])
    
    print("\n✓ 评估完成!")


if __name__ == '__main__':
    main()


模型评估: 计算所有checkpoint在验证集上的指标

加载验证集...
Dataset: for guided super-resolution.
📊 [validation] 原始样本数: 33,149
🎲 [validation] 随机采样 10,000 个样本...
✅ [validation] 采样后样本数: 10,000
Dataset [Dataset - validation_dataset] is created.
验证集: 10000 张图像

初始化模型结构...
Training model [ModelPlain] is created.
模型结构初始化完成

找到 16 个模型文件:
  - Step 16000: 16000_G.pth
  - Step 18000: 18000_G.pth
  - Step 32000: 32000_G.pth
  - Step 48000: 48000_G.pth
  - Step 64000: 64000_G.pth
  - Step 72000: 72000_G.pth
  - Step 80000: 80000_G.pth
  - Step 96000: 96000_G.pth
  - Step 112000: 112000_G.pth
  - Step 128000: 128000_G.pth
  - Step 144000: 144000_G.pth
  - Step 160000: 160000_G.pth
  - Step 176000: 176000_G.pth
  - Step 192000: 192000_G.pth
  - Step 208000: 208000_G.pth
  - Step 224000: 224000_G.pth

开始评估...

[Step 16000] 评估中...
  PSNR: 31.98 dB
  SSIM: 0.8644

[Step 18000] 评估中...
  PSNR: 31.91 dB
  SSIM: 0.8649

[Step 32000] 评估中...
  PSNR: 31.96 dB
  SSIM: 0.8766

[Step 48000] 评估中...
  PSNR: 32.40 dB
  SSIM: 0.8819

[S

In [7]:
#第二步：对验证样本生成结果
# --------------------------------------------------------
# Infrared Image Super-Resolution - Test Only Script
# Load Best Model and Generate Results (Optimized with Subset)
# --------------------------------------------------------
import os
import gc
import torch
from torch.utils.data import DataLoader, Subset  # ✅ 新增 Subset
from utils import utils_image as util
from utils import utils_option as option
from data.select_dataset import define_Dataset
from models.select_model import define_Model
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")


# ========================================
# 测试配置
# ========================================
def get_test_config():
    """测试配置"""
    config = {
        # 基本设置
        "task": "Guided SR",
        "model": "plain",
        "gpu_ids": [0],
        "dist": False,
        "scale": 8,
        "n_channels_lr": 1,
        "n_channels_guide": 1,
        
        # 路径设置
        "path": {
            "root": "F:/ChaoFen/Result/IR_VIS_SR_x8",
            "images": "F:/ChaoFen/Result/IR_VIS_SR_x8/4km_test_results",  # 测试结果保存路径
            "pretrained_netG": "F:/ChaoFen/Result/IR_VIS_SR_x8/models/224000_G.pth"  # 修改为你的最佳模型路径
        },
        
        # 测试数据集
        "datasets": {
            "test": {
                "name": "test_dataset",
                "dataset_type": "guided_sr",
                "phase": "test",
                "dataroot_lr": "F:/ChaoFen/Datasets/Val/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Val/visible/",
                "dataroot_gt": None,  # 如果没有GT可以设为None
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": False,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": None,
                "manual_seed": 60
            }
        },
        
        # 网络结构（必须与训练时一致）
       "netG": {
            "net_type": "swinfusionSR",
            "upscale": 8,
            "in_chans": 1,
            "img_size": 128,
            "window_size": 8,
            "img_range": 1.0,
            "embed_dim": 6,
            "Ex_depths": [2],
            "Ex_num_heads": [1],
            "Fusion_depths": [2],
            "Fusion_num_heads": [1],
            "Re_depths": [2],
            "Re_num_heads": [1],
            "mlp_ratio": 2,
            "upsampler": "",
            "resi_connection": "1conv",
            "init_type": "default"
        },
        
        # 训练设置
        "train": {
            "batch_size": 1,
            "G_lossfn_type": "l1",
            "weights": {
                "l1": 1.0,
                "mse": 0.0,
                "ssim": 0.0,
                "psnr": 0.0,
                "contrast": 0.0,
                "lpips": 0.0,
                "adversarial": 0.0
            },
            "manual_seed": 60,
            "G_lossfn_weight": 1.0,
            "E_decay": 0.999,
            "G_optimizer_type": "adam",
            "G_optimizer_lr": 1e-3,
            "G_optimizer_wd": 0,
            "G_optimizer_clipgrad": None,
            "G_optimizer_reuse": True,
            "G_scheduler_type": "MultiStepLR",
            "G_scheduler_milestones": [150000, 250000],
            "G_scheduler_gamma": 0.5,
            "G_regularizer_orthstep": None,
            "G_regularizer_clipstep": None,
            "G_param_strict": True,
            "E_param_strict": True,
            "checkpoint_test": 2000,
            "checkpoint_print": 2000,
            "limit_test": 1000000,
            "limit_validation": 1000000,
            "epochs": 200
        },
        
        "rank": 0,
        "world_size": 1,
        "num_gpu": 1
    }
    
    return config


def main():
    """主测试函数"""
    
    # ========================================
    # 配置参数
    # ========================================
    START_INDEX = 0  # ⚠️ 从第 179,000 个样本开始（修改这里）
    
    print("="*60)
    print("Infrared Image Super-Resolution - Test Only")
    print("="*60)
    
    # ----------------------------------------
    # 获取配置
    # ----------------------------------------
    opt = get_test_config()
    
    # 检查模型路径
    model_path = opt['path']['pretrained_netG']
    if not os.path.exists(model_path):
        print(f"❌ 错误: 模型文件不存在: {model_path}")
        print(f"\n请修改配置中的 'pretrained_netG' 路径为你的最佳模型路径")
        print(f"例如: F:/ChaoFen/Result/IR_VIS_SR_x8/models/10000_G.pth")
        return
    
    # 创建输出目录
    os.makedirs(opt['path']['images'], exist_ok=True)
    
    # 转换配置
    opt = option.dict_to_nonedict(opt)
    
    print(f"\n✓ 模型路径: {model_path}")
    print(f"✓ 输出路径: {opt['path']['images']}")
    
    # ----------------------------------------
    # 创建数据加载器（✅ 优化：使用 Subset）
    # ----------------------------------------
    print("\n加载测试数据集...")
    
    test_set = define_Dataset(opt['datasets']['test'])
    print(f"✓ 原始测试集: {len(test_set)} 张图像")
    
    # ✅ 关键优化：直接截取数据集
    if START_INDEX > 0:
        if START_INDEX >= len(test_set):
            print(f"❌ 错误: START_INDEX ({START_INDEX}) 超出数据集大小 ({len(test_set)})")
            return
        
        print(f"⚠️  从第 {START_INDEX} 个样本开始（跳过前 {START_INDEX} 个）")
        indices = list(range(START_INDEX, len(test_set)))
        test_set = Subset(test_set, indices)
        print(f"✓ 剩余样本数: {len(test_set)}")
    
    test_loader = DataLoader(
        test_set,
        batch_size=opt['train']['batch_size'],
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    # ----------------------------------------
    # 初始化模型并加载权重
    # ----------------------------------------
    print("\n初始化模型...")
    
    model = define_Model(opt)
    
    # 加载预训练权重
    print(f"加载模型权重: {model_path}")
    try:
        # 方法1: 尝试使用 param_key
        model.load_network(model_path, model.netG, strict=True, param_key='params')
        print("✓ 模型加载成功 (使用 param_key='params')")
    except:
        try:
            # 方法2: 不使用 param_key
            model.load_network(model_path, model.netG, strict=True)
            print("✓ 模型加载成功 (直接加载)")
        except Exception as e:
            print(f"❌ 模型加载失败: {e}")
            return
    
    # 设置为评估模式
    model.netG.eval()
    
    # ----------------------------------------
    # 测试循环（✅ 不再需要 if idx < START_INDEX）
    # ----------------------------------------
    print("\n" + "="*60)
    print("开始生成测试结果...")
    print("="*60)
    
    has_gt = opt['datasets']['test']['dataroot_gt'] is not None
    
    test_pbar = tqdm(test_loader, desc='测试进度', unit='img', ncols=100)
    
    with torch.no_grad():  # 不需要计算梯度
        for idx, test_data in enumerate(test_pbar):
            # ✅ 删除了这行！不再需要跳过
            # if idx < START_INDEX:
            #     continue
            
            # 前向传播
            model.feed_data(test_data, need_GT=has_gt, phase='test')
            model.test()
            
            # 获取输出
            visuals = model.current_visuals(need_H=has_gt)
            E_img_batch = visuals['Output']
            
            # 遍历 batch 中的每张图像
            batch_size = E_img_batch.shape[0]
            for i in range(batch_size):
                # 获取图像名称
                image_name = os.path.basename(test_data['Guide_path'][i])
                
                # 转换为 uint8
                E_img = util.tensor2uint(E_img_batch[i:i+1])
                
                # 保存图像
                save_path = os.path.join(opt['path']['images'], image_name)
                util.imsave(E_img, save_path)
                
                # 立即删除已保存的图像
                del E_img
            
            # 清理 batch 数据
            del E_img_batch, visuals, test_data
            
            # 清理 GPU 缓存
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            # 强制垃圾回收
            gc.collect()
            
            # 更新进度条（显示最后一张图像的名称）
            test_pbar.set_postfix({'最后': image_name})
    
    test_pbar.close()
    
    # ----------------------------------------
    # 输出结果
    # ----------------------------------------
    print("\n" + "="*60)
    print("测试完成!")
    print("="*60)
    print(f"✓ 结果已保存到: {opt['path']['images']}")
    print(f"✓ 共生成 {len(test_set)} 张图像")
    print("="*60)


if __name__ == '__main__':
    main()

Infrared Image Super-Resolution - Test Only

✓ 模型路径: F:/ChaoFen/Result/IR_VIS_SR_x8/models/224000_G.pth
✓ 输出路径: F:/ChaoFen/Result/IR_VIS_SR_x8/4km_test_results

加载测试数据集...
Dataset: for guided super-resolution.
Dataset [Dataset - test_dataset] is created.
✓ 原始测试集: 33149 张图像

初始化模型...
Training model [ModelPlain] is created.
加载模型权重: F:/ChaoFen/Result/IR_VIS_SR_x8/models/224000_G.pth
✓ 模型加载成功 (使用 param_key='params')

开始生成测试结果...


测试进度: 100%|█| 33149/33149 [3:53:41<00:00,  2.36img/s, 最后=20240417034500_20240417035959_7.68_12s]


测试完成!
✓ 结果已保存到: F:/ChaoFen/Result/IR_VIS_SR_x8/4km_test_results
✓ 共生成 33149 张图像


In [1]:
#第三步：对测试样本生成结果
# --------------------------------------------------------
# Infrared Image Super-Resolution - Test Only Script
# Load Best Model and Generate Results (Optimized with Subset)
# --------------------------------------------------------
import os
import gc
import torch
from torch.utils.data import DataLoader, Subset  # ✅ 新增 Subset
from utils import utils_image as util
from utils import utils_option as option
from data.select_dataset import define_Dataset
from models.select_model import define_Model
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")


# ========================================
# 测试配置
# ========================================
def get_test_config():
    """测试配置"""
    config = {
        # 基本设置
        "task": "Guided SR",
        "model": "plain",
        "gpu_ids": [0],
        "dist": False,
        "scale": 8,
        "n_channels_lr": 1,
        "n_channels_guide": 1,
        
        # 路径设置
        "path": {
            "root": "F:/ChaoFen/Result/IR_VIS_SR_x8",
            "images": "F:/ChaoFen/Result/IR_VIS_SR_x8/500m_test_results",  # 测试结果保存路径
            "pretrained_netG": "F:/ChaoFen/Result/IR_VIS_SR_x8/models/224000_G.pth"  # ⚠️ 修改为你的最佳模型路径
        },
        
        # 测试数据集
        "datasets": {
            "test": {
                "name": "test_dataset",
                "dataset_type": "guided_sr",
                "phase": "test",
                "dataroot_lr": "F:/ChaoFen/Datasets/Test/infrared/LR_x8/",
                "dataroot_guide": "F:/ChaoFen/Datasets/Test/visible/",
                "dataroot_gt": None,  # 如果没有GT可以设为None
                "H_size": 128,
                "proba_without_rgb": 0.0,
                "dataloader_shuffle": False,
                "patch": False,
                "dataloader_num_workers": 0,
                "n_channels_lr": 1,
                "n_channels_guide": 1,
                "scale": 8,
                "max_samples": None,
                "manual_seed": 60
            }
        },
        
        "netG": {
            "net_type": "swinfusionSR",
            "upscale": 8,
            "in_chans": 1,
            "img_size": 128,
            "window_size": 8,
            "img_range": 1.0,
            "embed_dim": 6,
            "Ex_depths": [2],
            "Ex_num_heads": [1],
            "Fusion_depths": [2],
            "Fusion_num_heads": [1],
            "Re_depths": [2],
            "Re_num_heads": [1],
            "mlp_ratio": 2,
            "upsampler": "",
            "resi_connection": "1conv",
            "init_type": "default"
        },
        
        # 训练设置
        "train": {
            "batch_size": 1,
            "G_lossfn_type": "l1",
            "weights": {
                "l1": 1.0,
                "mse": 0.0,
                "ssim": 0.0,
                "psnr": 0.0,
                "contrast": 0.0,
                "lpips": 0.0,
                "adversarial": 0.0
            },
            "manual_seed": 60,
            "G_lossfn_weight": 1.0,
            "E_decay": 0.999,
            "G_optimizer_type": "adam",
            "G_optimizer_lr": 1e-3,
            "G_optimizer_wd": 0,
            "G_optimizer_clipgrad": None,
            "G_optimizer_reuse": True,
            "G_scheduler_type": "MultiStepLR",
            "G_scheduler_milestones": [150000, 250000],
            "G_scheduler_gamma": 0.5,
            "G_regularizer_orthstep": None,
            "G_regularizer_clipstep": None,
            "G_param_strict": True,
            "E_param_strict": True,
            "checkpoint_test": 2000,
            "checkpoint_print": 2000,
            "limit_test": 1000000,
            "limit_validation": 1000000,
            "epochs": 200
        },
        
        "rank": 0,
        "world_size": 1,
        "num_gpu": 1
    }
    
    return config


def main():
    """主测试函数"""
    
    # ========================================
    # 配置参数
    # ========================================
    START_INDEX = 14000  # ⚠️ 从第 179,000 个样本开始（修改这里）
    
    print("="*60)
    print("Infrared Image Super-Resolution - Test Only")
    print("="*60)
    
    # ----------------------------------------
    # 获取配置
    # ----------------------------------------
    opt = get_test_config()
    
    # 检查模型路径
    model_path = opt['path']['pretrained_netG']
    if not os.path.exists(model_path):
        print(f"❌ 错误: 模型文件不存在: {model_path}")
        print(f"\n请修改配置中的 'pretrained_netG' 路径为你的最佳模型路径")
        print(f"例如: F:/ChaoFen/Result/IR_VIS_SR_x8/models/10000_G.pth")
        return
    
    # 创建输出目录
    os.makedirs(opt['path']['images'], exist_ok=True)
    
    # 转换配置
    opt = option.dict_to_nonedict(opt)
    
    print(f"\n✓ 模型路径: {model_path}")
    print(f"✓ 输出路径: {opt['path']['images']}")
    
    # ----------------------------------------
    # 创建数据加载器（✅ 优化：使用 Subset）
    # ----------------------------------------
    print("\n加载测试数据集...")
    
    test_set = define_Dataset(opt['datasets']['test'])
    print(f"✓ 原始测试集: {len(test_set)} 张图像")
    
    # ✅ 关键优化：直接截取数据集
    if START_INDEX > 0:
        if START_INDEX >= len(test_set):
            print(f"❌ 错误: START_INDEX ({START_INDEX}) 超出数据集大小 ({len(test_set)})")
            return
        
        print(f"⚠️  从第 {START_INDEX} 个样本开始（跳过前 {START_INDEX} 个）")
        indices = list(range(START_INDEX, len(test_set)))
        test_set = Subset(test_set, indices)
        print(f"✓ 剩余样本数: {len(test_set)}")
    
    test_loader = DataLoader(
        test_set,
        batch_size=opt['train']['batch_size'],
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    # ----------------------------------------
    # 初始化模型并加载权重
    # ----------------------------------------
    print("\n初始化模型...")
    
    model = define_Model(opt)
    
    # 加载预训练权重
    print(f"加载模型权重: {model_path}")
    try:
        # 方法1: 尝试使用 param_key
        model.load_network(model_path, model.netG, strict=True, param_key='params')
        print("✓ 模型加载成功 (使用 param_key='params')")
    except:
        try:
            # 方法2: 不使用 param_key
            model.load_network(model_path, model.netG, strict=True)
            print("✓ 模型加载成功 (直接加载)")
        except Exception as e:
            print(f"❌ 模型加载失败: {e}")
            return
    
    # 设置为评估模式
    model.netG.eval()
    
    # ----------------------------------------
    # 测试循环（✅ 不再需要 if idx < START_INDEX）
    # ----------------------------------------
    print("\n" + "="*60)
    print("开始生成测试结果...")
    print("="*60)
    
    has_gt = opt['datasets']['test']['dataroot_gt'] is not None
    
    test_pbar = tqdm(test_loader, desc='测试进度', unit='img', ncols=100)
    
    with torch.no_grad():  # 不需要计算梯度
        for idx, test_data in enumerate(test_pbar):
            # ✅ 删除了这行！不再需要跳过
            # if idx < START_INDEX:
            #     continue
            
            # 前向传播
            model.feed_data(test_data, need_GT=has_gt, phase='test')
            model.test()
            
            # 获取输出
            visuals = model.current_visuals(need_H=has_gt)
            E_img_batch = visuals['Output']
            
            # 遍历 batch 中的每张图像
            batch_size = E_img_batch.shape[0]
            for i in range(batch_size):
                # 获取图像名称
                image_name = os.path.basename(test_data['Guide_path'][i])
                
                # 转换为 uint8
                E_img = util.tensor2uint(E_img_batch[i:i+1])
                
                # 保存图像
                save_path = os.path.join(opt['path']['images'], image_name)
                util.imsave(E_img, save_path)
                
                # 立即删除已保存的图像
                del E_img
            
            # 清理 batch 数据
            del E_img_batch, visuals, test_data
            
            # 清理 GPU 缓存
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            # 强制垃圾回收
            gc.collect()
            
            # 更新进度条（显示最后一张图像的名称）
            test_pbar.set_postfix({'最后': image_name})
    
    test_pbar.close()
    
    # ----------------------------------------
    # 输出结果
    # ----------------------------------------
    print("\n" + "="*60)
    print("测试完成!")
    print("="*60)
    print(f"✓ 结果已保存到: {opt['path']['images']}")
    print(f"✓ 共生成 {len(test_set)} 张图像")
    print("="*60)


if __name__ == '__main__':
    main()


Infrared Image Super-Resolution - Test Only

✓ 模型路径: F:/ChaoFen/Result/IR_VIS_SR_x8/models/224000_G.pth
✓ 输出路径: F:/ChaoFen/Result/IR_VIS_SR_x8/500m_test_results

加载测试数据集...
Dataset: for guided super-resolution.
Dataset [Dataset - test_dataset] is created.
✓ 原始测试集: 154308 张图像
⚠️  从第 14000 个样本开始（跳过前 14000 个）
✓ 剩余样本数: 140308

初始化模型...
Training model [ModelPlain] is created.
加载模型权重: F:/ChaoFen/Result/IR_VIS_SR_x8/models/224000_G.pth
✓ 模型加载成功 (使用 param_key='params')

开始生成测试结果...


测试进度:  84%|▊| 117347/140308 [16:34:14<3:14:32,  1.97img/s, 最后=20240403063000_20240403064459_ros]


KeyboardInterrupt: 

In [1]:
from pathlib import Path

# 目标文件夹路径
folder_path = Path(r"F:\ChaoFen\Result\IR_VIS_SR_x8\500m_test_results")

# 统计 PNG 文件数量
png_files = list(folder_path.glob("*.png"))
png_count = len(png_files)

print("=" * 60)
print(f"文件夹: {folder_path}")
print("=" * 60)
print(f"PNG 文件数量: {png_count}")
print("=" * 60)

# 如果需要查看文件列表（可选）
if png_count > 0:
    print("\n前 10 个 PNG 文件:")
    for i, file in enumerate(png_files[:10], 1):
        print(f"  {i}. {file.name}")
        
    if png_count > 10:
        print(f"  ... 还有 {png_count - 10} 个文件")


文件夹: F:\ChaoFen\Result\IR_VIS_SR_x8\500m_test_results
PNG 文件数量: 193319

前 10 个 PNG 文件:
  1. 20240401010000_20240401011459_row038_col103_24.32_24.96_135.92_136.56.png
  2. 20240401010000_20240401011459_row038_col104_24.32_24.96_136.56_137.20.png
  3. 20240401010000_20240401011459_row038_col105_24.32_24.96_137.20_137.84.png
  4. 20240401010000_20240401011459_row038_col106_24.32_24.96_137.84_138.48.png
  5. 20240401010000_20240401011459_row038_col107_24.32_24.96_138.48_139.12.png
  6. 20240401010000_20240401011459_row038_col108_24.32_24.96_139.12_139.76.png
  7. 20240401010000_20240401011459_row039_col000_24.96_25.60_70.00_70.64.png
  8. 20240401010000_20240401011459_row039_col001_24.96_25.60_70.64_71.28.png
  9. 20240401010000_20240401011459_row039_col002_24.96_25.60_71.28_71.92.png
  10. 20240401010000_20240401011459_row039_col003_24.96_25.60_71.92_72.56.png
  ... 还有 193309 个文件


In [2]:
#第四步:将测试结果反归一化到亮温并拼接成完整的卫星红外影像(含经纬度)
import os
import re
import numpy as np
from PIL import Image
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import h5py
import gc  # ✅ 添加垃圾回收模块

# ============================================================================
# 配置参数
# ============================================================================

class Config:
    # 输入路径
    RESULTS_DIR = r"F:\ChaoFen\Result\IR_VIS_SR_x8\500m_test_results"  # 测试结果PNG目录
    NORM_PARAMS_PATH = r"F:\ChaoFen\Datasets\normalization_params.npz"  # 归一化参数
    
    # 输出路径
    OUTPUT_DIR = r"F:\ChaoFen\Result\IR_VIS_SR_x8\reconstructed_full_images"
    
    # 样本尺寸
    PATCH_SIZE = 128  # 每个patch的尺寸
    
    # 区域起点(边界像元中心)
    LAT_START = 0.0   # 起始纬度
    LON_START = 70.0  # 起始经度
    
    # 像元分辨率
    PIXEL_RESOLUTION = 0.005  # 度
    
    # 🔥 起始时间戳（跳过已完成的时间戳）
    # 格式: "20240403060000_20240403061459"
    # 设置为 None 表示从第一个时间戳开始处理
    START_TIMESTAMP = "20240401073000_20240401074459"  # 🔥 修改这里指定起始时间戳

# ============================================================================
# 工具函数
# ============================================================================

def parse_filename(filename):
    """
    解析文件名,提取时间戳和位置信息
    
    文件名格式: 20240403060000_20240403061459_row000_col000_10.00_10.64_70.00_70.64.png
    
    返回:
        dict: {
            'timestamp': '20240403060000_20240403061459',
            'row_idx': 0,
            'col_idx': 0,
            'lat_min': 10.00,
            'lat_max': 10.64,
            'lon_min': 70.00,
            'lon_max': 70.64
        }
    """
    pattern = r'(\d{14}_\d{14})_row(\d{3})_col(\d{3})_([\d.]+)_([\d.]+)_([\d.]+)_([\d.]+)\.png'
    match = re.match(pattern, filename)
    
    if match:
        return {
            'timestamp': match.group(1),
            'row_idx': int(match.group(2)),
            'col_idx': int(match.group(3)),
            'lat_min': float(match.group(4)),
            'lat_max': float(match.group(5)),
            'lon_min': float(match.group(6)),
            'lon_max': float(match.group(7))
        }
    else:
        return None

def load_normalization_params(params_path):
    """加载归一化参数"""
    params = np.load(params_path)
    return {
        'hr_ir_min': float(params['hr_ir_min']),
        'hr_ir_max': float(params['hr_ir_max'])
    }

def denormalize_image(img_uint8, data_min, data_max):
    """
    反归一化图像
    
    参数:
        img_uint8: uint8类型的图像 (0-255)
        data_min: 原始数据最小值
        data_max: 原始数据最大值
    
    返回:
        原始亮温值 (float32)
    """
    # 归一化到 [0, 1]
    normalized = img_uint8.astype(np.float32) / 255.0
    
    # 反归一化到原始范围
    original = normalized * (data_max - data_min) + data_min
    
    return original

def group_files_by_timestamp(results_dir):
    """
    按时间戳分组文件
    
    返回:
        dict: {timestamp: [file_info1, file_info2, ...]}
    """
    results_dir = Path(results_dir)
    png_files = sorted(results_dir.glob("*.png"))
    
    grouped = defaultdict(list)
    
    for png_file in png_files:
        info = parse_filename(png_file.name)
        if info:
            info['file_path'] = str(png_file)
            grouped[info['timestamp']].append(info)
        else:
            print(f"⚠️  无法解析文件名: {png_file.name}")
    
    return grouped

def filter_timestamps_from_start(grouped_files, start_timestamp):
    """
    从指定时间戳开始过滤
    
    参数:
        grouped_files: 按时间戳分组的文件字典
        start_timestamp: 起始时间戳（包含该时间戳）
    
    返回:
        过滤后的字典
    """
    if start_timestamp is None:
        return grouped_files
    
    # 获取所有时间戳并排序
    all_timestamps = sorted(grouped_files.keys())
    
    # 检查起始时间戳是否存在
    if start_timestamp not in all_timestamps:
        print(f"⚠️  警告: 起始时间戳 {start_timestamp} 不存在于数据中")
        print(f"   可用的时间戳范围: {all_timestamps[0]} 到 {all_timestamps[-1]}")
        print(f"   将从第一个时间戳开始处理")
        return grouped_files
    
    # 找到起始时间戳的索引
    start_idx = all_timestamps.index(start_timestamp)
    
    # 过滤出从起始时间戳开始的所有时间戳
    filtered_timestamps = all_timestamps[start_idx:]
    filtered_grouped = {ts: grouped_files[ts] for ts in filtered_timestamps}
    
    print(f"✂️  时间戳过滤:")
    print(f"   总时间戳数: {len(all_timestamps)}")
    print(f"   起始时间戳: {start_timestamp}")
    print(f"   跳过时间戳数: {start_idx}")
    print(f"   待处理时间戳数: {len(filtered_grouped)}")
    
    return filtered_grouped

def create_coordinate_grids(n_rows, n_cols, patch_size=128):
    """
    创建完整的经纬度网格
    
    参数:
        n_rows: 行数(patch数量)
        n_cols: 列数(patch数量)
        patch_size: 每个patch的像元数量
    
    返回:
        lon_grid: 经度网格 (2D array)
        lat_grid: 纬度网格 (2D array)
    
    计算公式:
        对于 row_idx=r, col_idx=c 的patch:
        - 第一行第一列像元纬度 = LAT_START + r * 16 * 8 * 0.005
        - 第一行第一列像元经度 = LON_START + c * 16 * 8 * 0.005
        - 其余像元逐行列递增 0.005
    """
    full_height = n_rows * patch_size
    full_width = n_cols * patch_size
    
    # 初始化网格
    lon_grid = np.zeros((full_height, full_width), dtype=np.float32)
    lat_grid = np.zeros((full_height, full_width), dtype=np.float32)
    
    # 为每个patch计算经纬度
    for r in range(n_rows):
        for c in range(n_cols):
            # 计算该patch第一行第一列像元的经纬度
            # r * 16 * 8 = r * 128 (因为16个4km像元 × 8倍上采样 = 128个500m像元)
            lat_start = Config.LAT_START + r * 16 * 8 * Config.PIXEL_RESOLUTION
            lon_start = Config.LON_START + c * 16 * 8 * Config.PIXEL_RESOLUTION
            
            # 生成该patch内所有像元的经纬度
            # 行索引 i: 0 到 127, 列索引 j: 0 到 127
            for i in range(patch_size):
                for j in range(patch_size):
                    # 全局索引
                    global_i = r * patch_size + i
                    global_j = c * patch_size + j
                    
                    # 经纬度递增
                    lat_grid[global_i, global_j] = lat_start + i * Config.PIXEL_RESOLUTION
                    lon_grid[global_i, global_j] = lon_start + j * Config.PIXEL_RESOLUTION
    
    # 保留3位小数
    lat_grid = np.round(lat_grid, 3)
    lon_grid = np.round(lon_grid, 3)
    
    return lon_grid, lat_grid

def reconstruct_full_image(file_infos, norm_params, patch_size=128):
    """重建完整图像"""
    # 1. 确定网格大小
    max_row = max(info['row_idx'] for info in file_infos)
    max_col = max(info['col_idx'] for info in file_infos)
    n_rows = max_row + 1
    n_cols = max_col + 1
    
    print(f"  网格大小: {n_rows} 行 × {n_cols} 列")
    print(f"  开始拼接 {len(file_infos)} 个patches...")
    
    # 2. 初始化完整图像矩阵
    full_height = n_rows * patch_size
    full_width = n_cols * patch_size
    full_image = np.full((full_height, full_width), np.nan, dtype=np.float32)
    
    # 3. 逐个读取patch并放置到正确位置
    for idx, info in enumerate(file_infos):
        # 每处理1000个打印一次进度
        if (idx + 1) % 1000 == 0:
            print(f"    已处理 {idx + 1}/{len(file_infos)} patches")
        
        # 读取PNG图像
        img = Image.open(info['file_path'])
        img_array = np.array(img)
        
        # 反归一化
        brightness_temp = denormalize_image(
            img_array, 
            norm_params['hr_ir_min'], 
            norm_params['hr_ir_max']
        )
        
        # 计算在完整图像中的位置
        row_start = info['row_idx'] * patch_size
        row_end = row_start + patch_size
        col_start = info['col_idx'] * patch_size
        col_end = col_start + patch_size
        
        # 放置到完整图像中
        full_image[row_start:row_end, col_start:col_end] = brightness_temp
    
    print(f"  ✓ 完成拼接")
    return full_image, (n_rows, n_cols)


def save_as_hdf5(full_image, lon_grid, lat_grid, timestamp, output_dir):
    """
    保存为HDF5格式(包含经纬度)
    
    参数:
        full_image: 完整的亮温矩阵
        lon_grid: 经度网格
        lat_grid: 纬度网格
        timestamp: 时间戳 (格式: 20240403060000_20240403061459)
        output_dir: 输出目录
    
    返回:
        保存的文件路径
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 文件名: 20240403060000_20240403061459.hdf5
    hdf5_path = output_dir / f"{timestamp}.hdf5"
    
    with h5py.File(hdf5_path, 'w') as f:
        # 创建亮温数据集
        bt_dataset = f.create_dataset(
            'brightness_temperature', 
            data=full_image,
            compression='gzip',
            compression_opts=4
        )
        bt_dataset.attrs['unit'] = 'Kelvin'
        bt_dataset.attrs['long_name'] = 'Infrared brightness temperature'
        bt_dataset.attrs['description'] = 'Reconstructed 500m infrared brightness temperature'
        
        # 创建经度数据集
        lon_dataset = f.create_dataset(
            'longitude',
            data=lon_grid,
            compression='gzip',
            compression_opts=4
        )
        lon_dataset.attrs['unit'] = 'degrees_east'
        lon_dataset.attrs['long_name'] = 'Longitude'
        lon_dataset.attrs['description'] = 'Pixel center longitude (3 decimal places)'
        
        # 创建纬度数据集
        lat_dataset = f.create_dataset(
            'latitude',
            data=lat_grid,
            compression='gzip',
            compression_opts=4
        )
        lat_dataset.attrs['unit'] = 'degrees_north'
        lat_dataset.attrs['long_name'] = 'Latitude'
        lat_dataset.attrs['description'] = 'Pixel center latitude (3 decimal places)'
        
        # 全局属性
        f.attrs['timestamp'] = timestamp
        f.attrs['shape'] = full_image.shape
        f.attrs['resolution'] = '500m (0.005 degree)'
        f.attrs['scale'] = 8  # 超分辨率倍数
        f.attrs['coordinate_precision'] = '3 decimal places'
        f.attrs['lat_start'] = Config.LAT_START
        f.attrs['lon_start'] = Config.LON_START
        f.attrs['pixel_resolution'] = Config.PIXEL_RESOLUTION
    
    return str(hdf5_path)

# ============================================================================
# 主程序
# ============================================================================

def main():
    
    print("="*80)
    print("start")
    print("="*80)
    
    # 1. 加载归一化参数
    print("\n[1/5] 加载归一化参数...")
    norm_params = load_normalization_params(Config.NORM_PARAMS_PATH)
    print(f"  HR_IR范围: [{norm_params['hr_ir_min']:.4f}, {norm_params['hr_ir_max']:.4f}] K")
    
    # 2. 按时间戳分组文件
    print("\n[2/5] 扫描测试结果...")
    grouped_files = group_files_by_timestamp(Config.RESULTS_DIR)
    print(f"  找到 {len(grouped_files)} 个时间戳")
    
    if len(grouped_files) == 0:
        print("❌ 未找到任何测试结果文件!")
        return
    
    # 打印统计信息
    for timestamp, files in list(grouped_files.items())[:3]:
        print(f"    {timestamp}: {len(files)} 个patches")
    if len(grouped_files) > 3:
        print(f"    ... (共 {len(grouped_files)} 个时间戳)")
    
    # 🔥 3. 从指定时间戳开始过滤
    print("\n[3/5] 过滤时间戳...")
    if Config.START_TIMESTAMP:
        grouped_files = filter_timestamps_from_start(grouped_files, Config.START_TIMESTAMP)
        if len(grouped_files) == 0:
            print("❌ 过滤后没有待处理的时间戳!")
            return
    else:
        print("  未指定起始时间戳，将处理所有时间戳")
    
    # 4. 逐个时间戳重建完整图像
    print("\n[4/5] 重建完整图像...")
    
    saved_files = []
    
    for timestamp, file_infos in tqdm(grouped_files.items(), desc="处理时间戳"):
        print(f"\n  时间戳: {timestamp} ({len(file_infos)} patches)")
        
        # 重建完整图像
        full_image, grid_shape = reconstruct_full_image(
            file_infos, 
            norm_params, 
            patch_size=Config.PATCH_SIZE
        )
        
        n_rows, n_cols = grid_shape
        
        # 生成经纬度网格
        print("  生成经纬度网格...")
        lon_grid, lat_grid = create_coordinate_grids(n_rows, n_cols, Config.PATCH_SIZE)
        
        print(f"  完整图像尺寸: {full_image.shape}")
        print(f"  亮温范围: [{np.nanmin(full_image):.2f}, {np.nanmax(full_image):.2f}] K")
        print(f"  经度范围: [{lon_grid.min():.3f}, {lon_grid.max():.3f}]°E")
        print(f"  纬度范围: [{lat_grid.min():.3f}, {lat_grid.max():.3f}]°N")
        print(f"  有效像素: {np.sum(~np.isnan(full_image)):,} / {full_image.size:,}")
        
        # 保存为HDF5(含经纬度)
        hdf5_file = save_as_hdf5(full_image, lon_grid, lat_grid, timestamp, Config.OUTPUT_DIR)
        saved_files.append(hdf5_file)
        print(f"  ✓ 已保存: {hdf5_file}")
        
        # ✅ 清理内存缓存
        del full_image, lon_grid, lat_grid
        gc.collect()
        print(f"  🗑️  已清理内存缓存")
    
    # 5. 输出统计信息
    print("\n" + "="*80)
    print("✅ 重建完成!")
    print("="*80)
    print(f"处理时间戳数: {len(grouped_files)}")
    print(f"处理patches数: {sum(len(files) for files in grouped_files.values()):,}")
    print(f"保存文件数: {len(saved_files)}")
    print(f"\n输出目录: {Config.OUTPUT_DIR}")
    
    if Config.START_TIMESTAMP:
        print(f"\n⏭️  起始时间戳: {Config.START_TIMESTAMP}")
    
    print("\n" + "="*80)
    print("📖 使用方法:")
    print("="*80)

if __name__ == "__main__":
    main()


start

[1/5] 加载归一化参数...
  HR_IR范围: [192.8700, 304.8000] K

[2/5] 扫描测试结果...
  找到 14 个时间戳
    20240401023000_20240401024459: 9265 个patches
    20240401040000_20240401041459: 9264 个patches
    20240401053000_20240401054459: 9265 个patches
    ... (共 14 个时间戳)

[3/5] 过滤时间戳...
✂️  时间戳过滤:
   总时间戳数: 14
   起始时间戳: 20240401073000_20240401074459
   跳过时间戳数: 5
   待处理时间戳数: 9

[4/5] 重建完整图像...


处理时间戳:   0%|                                                                                | 0/9 [00:00<?, ?it/s]


  时间戳: 20240401073000_20240401074459 (9264 patches)
  网格大小: 85 行 × 109 列
  开始拼接 9264 个patches...
    已处理 1000/9264 patches
    已处理 2000/9264 patches
    已处理 3000/9264 patches
    已处理 4000/9264 patches
    已处理 5000/9264 patches
    已处理 6000/9264 patches
    已处理 7000/9264 patches
    已处理 8000/9264 patches
    已处理 9000/9264 patches
  ✓ 完成拼接
  生成经纬度网格...
  完整图像尺寸: (10880, 13952)
  亮温范围: [192.87, 304.80] K
  经度范围: [70.000, 139.755]°E
  纬度范围: [0.000, 54.395]°N
  有效像素: 151,781,376 / 151,797,760


处理时间戳:  11%|███████▋                                                             | 1/9 [10:55<1:27:21, 655.18s/it]

  ✓ 已保存: F:\ChaoFen\Result\IR_VIS_SR_x8\reconstructed_full_images\20240401073000_20240401074459.hdf5
  🗑️  已清理内存缓存

  时间戳: 20240402033000_20240402034459 (9265 patches)
  网格大小: 85 行 × 109 列
  开始拼接 9265 个patches...
    已处理 1000/9265 patches
    已处理 2000/9265 patches
    已处理 3000/9265 patches
    已处理 4000/9265 patches
    已处理 5000/9265 patches
    已处理 6000/9265 patches
    已处理 7000/9265 patches
    已处理 8000/9265 patches
    已处理 9000/9265 patches
  ✓ 完成拼接
  生成经纬度网格...
  完整图像尺寸: (10880, 13952)
  亮温范围: [192.87, 304.80] K
  经度范围: [70.000, 139.755]°E
  纬度范围: [0.000, 54.395]°N
  有效像素: 151,797,760 / 151,797,760
  ✓ 已保存: F:\ChaoFen\Result\IR_VIS_SR_x8\reconstructed_full_images\20240402033000_20240402034459.hdf5


处理时间戳:  22%|███████████████▎                                                     | 2/9 [24:11<1:26:08, 738.33s/it]

... (11 outputs left out)

处理时间戳:  89%|████████████████████████████████████████████████████████████▍       | 8/9 [2:06:38<17:48, 1068.02s/it]


  时间戳: 20240403050000_20240403051459 (9261 patches)
  网格大小: 85 行 × 109 列
  开始拼接 9261 个patches...
    已处理 1000/9261 patches
    已处理 2000/9261 patches
    已处理 3000/9261 patches
    已处理 4000/9261 patches
    已处理 5000/9261 patches
    已处理 6000/9261 patches
    已处理 7000/9261 patches
    已处理 8000/9261 patches
    已处理 9000/9261 patches
  ✓ 完成拼接
  生成经纬度网格...
  完整图像尺寸: (10880, 13952)
  亮温范围: [192.87, 304.80] K
  经度范围: [70.000, 139.755]°E
  纬度范围: [0.000, 54.395]°N
  有效像素: 151,732,224 / 151,797,760


处理时间戳: 100%|█████████████████████████████████████████████████████████████████████| 9/9 [2:29:09<00:00, 994.43s/it]

  ✓ 已保存: F:\ChaoFen\Result\IR_VIS_SR_x8\reconstructed_full_images\20240403050000_20240403051459.hdf5
  🗑️  已清理内存缓存

✅ 重建完成!
处理时间戳数: 9
处理patches数: 83,375
保存文件数: 9

输出目录: F:\ChaoFen\Result\IR_VIS_SR_x8\reconstructed_full_images

⏭️  起始时间戳: 20240401073000_20240401074459

📖 使用方法:


In [1]:
#第五步：验证HDF5文件中第一行像元的经纬度
"""
读取已生成的HDF5文件，检查第一行像元的经纬度是否正确
"""

import h5py
import numpy as np
from pathlib import Path

# ============================================================================
# 配置参数
# ============================================================================

class Config:
    # HDF5文件目录
    HDF5_DIR = r"F:\ChaoFen\Result\IR_VIS_SR_x8\reconstructed_full_images"
    
    # 预期参数（用于验证）
    LAT_START = 0.0
    LON_START = 70.0
    PIXEL_RESOLUTION = 0.005

# ============================================================================
# 验证函数
# ============================================================================

def verify_hdf5_coordinates(hdf5_file):
    """
    验证HDF5文件中第一行像元的经纬度
    
    参数:
        hdf5_file: HDF5文件路径
    """
    print("\n" + "="*80)
    print("🔍 验证HDF5文件")
    print("="*80)
    print(f"文件: {Path(hdf5_file).name}\n")
    
    with h5py.File(hdf5_file, 'r') as f:
        # 读取数据
        lon = f['longitude'][:]
        lat = f['latitude'][:]
        bt = f['brightness_temperature'][:]
        
        print(f"数据形状: {bt.shape}")
        print(f"经度网格形状: {lon.shape}")
        print(f"纬度网格形状: {lat.shape}")
        
        # ========== 1. 检查第一行数据 ==========
        print("\n" + "="*80)
        print("第一行像元（前10个）:")
        print("="*80)
        print(f"{'列索引':<8} {'经度(°E)':<12} {'纬度(°N)':<12} {'亮温(K)':<12}")
        print("-" * 50)
        
        first_row_lon = lon[0, :]
        first_row_lat = lat[0, :]
        first_row_bt = bt[0, :]
        
        for i in range(min(10, len(first_row_lon))):
            print(f"{i:<8} {first_row_lon[i]:<12.3f} {first_row_lat[i]:<12.3f} {first_row_bt[i]:<12.2f}")
        
        # ========== 2. 检查第一行最后10个像元 ==========
        if len(first_row_lon) > 20:
            print("\n第一行像元（最后10个）:")
            print(f"{'列索引':<8} {'经度(°E)':<12} {'纬度(°N)':<12} {'亮温(K)':<12}")
            print("-" * 50)
            for i in range(max(0, len(first_row_lon) - 10), len(first_row_lon)):
                print(f"{i:<8} {first_row_lon[i]:<12.3f} {first_row_lat[i]:<12.3f} {first_row_bt[i]:<12.2f}")
        
        # ========== 3. 统计信息 ==========
        print("\n" + "="*80)
        print("第一行统计信息:")
        print("="*80)
        print(f"像元数量: {len(first_row_lon)}")
        print(f"经度范围: [{first_row_lon.min():.3f}, {first_row_lon.max():.3f}]°E")
        print(f"纬度范围: [{first_row_lat.min():.3f}, {first_row_lat.max():.3f}]°N")
        print(f"亮温范围: [{np.nanmin(first_row_bt):.2f}, {np.nanmax(first_row_bt):.2f}] K")
        print(f"有效像元: {np.sum(~np.isnan(first_row_bt))} / {len(first_row_bt)}")
        
        # ========== 4. 验证纬度一致性 ==========
        print("\n" + "="*80)
        print("纬度一致性检查:")
        print("="*80)
        lat_unique = np.unique(first_row_lat)
        if len(lat_unique) == 1:
            print(f"✅ 第一行纬度一致: {lat_unique[0]:.3f}°N")
        else:
            print(f"⚠️  第一行纬度不一致! 有 {len(lat_unique)} 个不同值:")
            print(f"   {lat_unique[:10]}")
        
        # ========== 5. 验证经度递增 ==========
        print("\n" + "="*80)
        print("经度递增检查:")
        print("="*80)
        lon_diff = np.diff(first_row_lon)
        lon_diff_unique = np.unique(np.round(lon_diff, 5))
        
        if len(lon_diff_unique) == 1:
            print(f"✅ 第一行经度均匀递增")
            print(f"   间隔: {lon_diff_unique[0]:.5f}°")
        else:
            print(f"⚠️  第一行经度递增不均匀! 有 {len(lon_diff_unique)} 个不同间隔:")
            print(f"   {lon_diff_unique}")
        
        # ========== 6. 与预期值对比 ==========
        print("\n" + "="*80)
        print("预期值验证:")
        print("="*80)
        
        # 第一行第一列纬度
        print(f"第一行第一列纬度:")
        print(f"  预期: {Config.LAT_START:.3f}°N")
        print(f"  实际: {first_row_lat[0]:.3f}°N")
        diff_lat = abs(first_row_lat[0] - Config.LAT_START)
        print(f"  差异: {diff_lat:.6f}°")
        if diff_lat < 0.001:
            print(f"  ✅ 符合预期")
        else:
            print(f"  ⚠️  与预期不符")
        
        # 第一行第一列经度
        print(f"\n第一行第一列经度:")
        print(f"  预期: {Config.LON_START:.3f}°E")
        print(f"  实际: {first_row_lon[0]:.3f}°E")
        diff_lon = abs(first_row_lon[0] - Config.LON_START)
        print(f"  差异: {diff_lon:.6f}°")
        if diff_lon < 0.001:
            print(f"  ✅ 符合预期")
        else:
            print(f"  ⚠️  与预期不符")
        
        # 像元分辨率
        if len(lon_diff_unique) == 1:
            print(f"\n像元分辨率:")
            print(f"  预期: {Config.PIXEL_RESOLUTION:.5f}°")
            print(f"  实际: {lon_diff_unique[0]:.5f}°")
            diff_res = abs(lon_diff_unique[0] - Config.PIXEL_RESOLUTION)
            print(f"  差异: {diff_res:.6f}°")
            if diff_res < 0.00001:
                print(f"  ✅ 符合预期")
            else:
                print(f"  ⚠️  与预期不符")
        
        # ========== 7. 检查第一列数据 ==========
        print("\n" + "="*80)
        print("第一列像元（前10个）:")
        print("="*80)
        print(f"{'行索引':<8} {'经度(°E)':<12} {'纬度(°N)':<12} {'亮温(K)':<12}")
        print("-" * 50)
        
        first_col_lon = lon[:, 0]
        first_col_lat = lat[:, 0]
        first_col_bt = bt[:, 0]
        
        for i in range(min(10, len(first_col_lat))):
            print(f"{i:<8} {first_col_lon[i]:<12.3f} {first_col_lat[i]:<12.3f} {first_col_bt[i]:<12.2f}")
        
        # 验证第一列经度一致性
        print("\n第一列经度一致性检查:")
        lon_unique_col = np.unique(first_col_lon)
        if len(lon_unique_col) == 1:
            print(f"✅ 第一列经度一致: {lon_unique_col[0]:.3f}°E")
        else:
            print(f"⚠️  第一列经度不一致! 有 {len(lon_unique_col)} 个不同值")
        
        # 验证第一列纬度递增
        print("\n第一列纬度递增检查:")
        lat_diff = np.diff(first_col_lat)
        lat_diff_unique = np.unique(np.round(lat_diff, 5))
        
        if len(lat_diff_unique) == 1:
            print(f"✅ 第一列纬度均匀递增")
            print(f"   间隔: {lat_diff_unique[0]:.5f}°")
        else:
            print(f"⚠️  第一列纬度递增不均匀! 有 {len(lat_diff_unique)} 个不同间隔:")
            print(f"   {lat_diff_unique}")
        
        # ========== 8. 文件属性 ==========
        print("\n" + "="*80)
        print("文件属性:")
        print("="*80)
        for key, value in f.attrs.items():
            print(f"{key}: {value}")
        
        # ========== 9. 全局统计 ==========
        print("\n" + "="*80)
        print("全局统计:")
        print("="*80)
        print(f"完整图像尺寸: {bt.shape}")
        print(f"经度范围: [{lon.min():.3f}, {lon.max():.3f}]°E")
        print(f"纬度范围: [{lat.min():.3f}, {lat.max():.3f}]°N")
        print(f"亮温范围: [{np.nanmin(bt):.2f}, {np.nanmax(bt):.2f}] K")
        print(f"有效像素: {np.sum(~np.isnan(bt)):,} / {bt.size:,}")
        print(f"覆盖率: {np.sum(~np.isnan(bt)) / bt.size * 100:.2f}%")

# ============================================================================
# 主程序
# ============================================================================

def main():
    print("="*80)
    print("HDF5文件经纬度验证工具")
    print("="*80)
    
    # 扫描HDF5文件
    hdf5_dir = Path(Config.HDF5_DIR)
    hdf5_files = sorted(hdf5_dir.glob("*.hdf5"))
    
    if len(hdf5_files) == 0:
        print(f"❌ 未找到HDF5文件: {hdf5_dir}")
        return
    
    print(f"\n找到 {len(hdf5_files)} 个HDF5文件")
    print(f"将验证第一个文件: {hdf5_files[0].name}")
    
    # 验证第一个文件
    verify_hdf5_coordinates(hdf5_files[0])
    
    # 询问是否验证更多文件
    if len(hdf5_files) > 1:
        print("\n" + "="*80)
        print(f"是否验证其他文件? (共 {len(hdf5_files)} 个)")
        print("输入文件索引 (0-{}), 或输入 'all' 验证所有文件, 或按 Enter 退出".format(len(hdf5_files)-1))
        
        user_input = input(">>> ").strip()
        
        if user_input.lower() == 'all':
            for i, hdf5_file in enumerate(hdf5_files[1:], 1):
                print(f"\n验证文件 [{i+1}/{len(hdf5_files)}]:")
                verify_hdf5_coordinates(hdf5_file)
        elif user_input.isdigit():
            idx = int(user_input)
            if 0 <= idx < len(hdf5_files):
                verify_hdf5_coordinates(hdf5_files[idx])
            else:
                print(f"❌ 索引超出范围 (0-{len(hdf5_files)-1})")
    
    print("\n" + "="*80)
    print("✅ 验证完成")
    print("="*80)

if __name__ == "__main__":
    main()


HDF5文件经纬度验证工具

找到 19 个HDF5文件
将验证第一个文件: 20240401010000_20240401011459.hdf5

🔍 验证HDF5文件
文件: 20240401010000_20240401011459.hdf5

数据形状: (10880, 13952)
经度网格形状: (10880, 13952)
纬度网格形状: (10880, 13952)

第一行像元（前10个）:
列索引      经度(°E)       纬度(°N)       亮温(K)       
--------------------------------------------------
0        70.000       0.000        286.80      
1        70.005       0.000        287.68      
2        70.010       0.000        287.24      
3        70.015       0.000        287.24      
4        70.020       0.000        287.68      
5        70.025       0.000        287.24      
6        70.030       0.000        287.24      
7        70.035       0.000        287.24      
8        70.040       0.000        287.24      
9        70.045       0.000        287.68      

第一行像元（最后10个）:
列索引      经度(°E)       纬度(°N)       亮温(K)       
--------------------------------------------------
13942    139.710      0.000        288.12      
13943    139.715      0.000        288.12      
1394

KeyboardInterrupt: Interrupted by user